In [ ]:
# In this section I am importing all the libraries I will need
import numpy as np
import matplotlib.pyplot as pl

In [ ]:
# In this section I am defining variables (arrays) that I would need to implement the method

# physical parameters
L = 0.05              # blade length, m
k = 25.0              # thermal conductivity, W/m/K
A = 2.0e-4            # cross-sectional area, m^2
P = 0.08              # wetted perimeter, m
hconv = 350.0         # convection coefficient, W/m^2/K
Tinf = 1400.0         # surrounding gas temperature, K
Tb = 900.0            # blade root temperature, K
TL = 1050.0           # blade tip temperature, K

# fin parameter
m = np.sqrt(hconv * P / (k * A))

# number of subintervals
N = 40

# Analytical solution for validation

def T_analytical(x, L, m, Tinf, Tb, TL):
    thetab = Tb - Tinf
    thetaL = TL - Tinf
    return Tinf + thetab*np.cosh(m*x) + (thetaL - thetab*np.cosh(m*L))*np.sinh(m*x)/np.sinh(m*L)

In [ ]:
# In this section I am setting the boundary conditions/initial values



In [ ]:
# In this section I am implementing the numerical method

# Finite difference direct method
def FiniteDiffBlade(L, m, Tinf, Tb, TL, N):
    # define the nodal points
    x = np.linspace(0, L, N+1)
    # determine the node spacing
    h = x[1] - x[0]

    # build A*T = b
    A = np.zeros((N+1, N+1))
    b = np.zeros(N+1)

    # boundary conditions
    A[0,0] = 1.0
    b[0] = Tb
    A[N,N] = 1.0
    b[N] = TL

    # interior equations
    for i in range(1, N):
        A[i, i-1] = 1.0
        A[i, i]   = -2.0 - m**2 * h**2
        A[i, i+1] = 1.0
        b[i] = -m**2 * h**2 * Tinf

    # direct matrix solve
    T = np.linalg.solve(A, b)
    return (x, T, A, b, h)

(xn, Tn, A, b, hstep) = FiniteDiffBlade(L, m, Tinf, Tb, TL, N)
Tan = T_analytical(xn, L, m, Tinf, Tb, TL)

print('m =', m)
print('h =', hstep)
print('maximum absolute error =', np.max(np.abs(Tn - Tan)))

# Gauss-Seidel iterative method

def GaussSeidelBlade(L, m, Tinf, Tb, TL, N, tol=1.0e-10, maxit=20000):
    x = np.linspace(0, L, N+1)
    h = x[1] - x[0]

    T = np.linspace(Tb, TL, N+1)
    residual_hist = []

    for kiter in range(maxit):
        Told = T.copy()

        # enforce boundary conditions
        T[0] = Tb
        T[N] = TL

        # Gauss-Seidel update for interior nodes
        for i in range(1, N):
            T[i] = (T[i-1] + Told[i+1] + m**2 * h**2 * Tinf) / (2.0 + m**2 * h**2)

        res = np.max(np.abs(T - Told))
        residual_hist.append(res)

        if res < tol:
            break

    return (x, T, np.array(residual_hist), kiter+1)

(xgs, Tgs, res_hist, niter) = GaussSeidelBlade(L, m, Tinf, Tb, TL, N)

print('Gauss-Seidel iterations =', niter)
print('direct / GS maximum difference =', np.max(np.abs(Tn - Tgs)))

In [ ]:
# In this section I am doing any post-processing (if needed)

#Post-processing: numerical differentiation for heat flux

def HeatFlux(x, T, k):
    h = x[1] - x[0]
    q = np.zeros_like(T)

    # end points
    q[0] = -k * (T[1] - T[0]) / h
    q[-1] = -k * (T[-1] - T[-2]) / h

    # interior nodes: central difference
    for i in range(1, len(T)-1):
        q[i] = -k * (T[i+1] - T[i-1]) / (2*h)

    return q

qn = HeatFlux(xn, Tn, k)
qan = HeatFlux(xn, Tan, k)


# Grid refinement study

# Because this is a steady BVP rather than a time-marching method, the reliability is assessed using a grid refinement study.
# The exact solution makes it possible to measure the numerical error directly.

Nvals = np.array([10, 20, 40, 80, 160, 320])
err = np.zeros(len(Nvals))

for j in range(len(Nvals)):
    (xj, Tj, Aj, bj, hj) = FiniteDiffBlade(L, m, Tinf, Tb, TL, int(Nvals[j]))
    Taj = T_analytical(xj, L, m, Tinf, Tb, TL)
    err[j] = np.max(np.abs(Tj - Taj))

# estimate order from final two refinements
p_est = np.log(err[-2]/err[-1]) / np.log((L/Nvals[-2])/(L/Nvals[-1]))
print('estimated order of accuracy =', p_est)

In [ ]:
# In this section I am showing/plotting the results


# Three different plots are included:
# 1. temperature distribution along the blade,
# 2. Gauss-Seidel residual convergence,
# 3. heat-flux distribution,
# 4. grid-refinement error plot.


# Plot 1: temperature distribution
pl.figure(figsize=(8,5))
pl.plot(xn, Tan, 'k-', label='Analytical')
pl.plot(xn, Tn, 'ro', fillstyle='none', label='Finite difference')
pl.plot(xgs, Tgs, 'b--', label='Gauss-Seidel')
pl.xlabel('x (m)')
pl.ylabel('T (K)')
pl.title('Temperature distribution along the blade')
pl.grid()
pl.legend()
pl.show()

# Plot 2: Gauss-Seidel residual convergence
pl.figure(figsize=(8,5))
pl.semilogy(np.arange(1, len(res_hist)+1), res_hist)
pl.xlabel('Iteration')
pl.ylabel('max |T^(k+1) - T^(k)|')
pl.title('Gauss-Seidel convergence history')
pl.grid()
pl.show()

# Plot 3: heat flux distribution
pl.figure(figsize=(8,5))
pl.plot(xn, qn, 'g-', label='Numerical heat flux')
pl.plot(xn, qan, 'k--', label='Analytical heat flux estimate')
pl.xlabel('x (m)')
pl.ylabel('q_x (W/m^2)')
pl.title('Heat flux along the blade')
pl.grid()
pl.legend()
pl.show()

# Plot 4: grid refinement
pl.figure(figsize=(8,5))
pl.loglog(L/Nvals, err, 'o-')
pl.xlabel('h (m)')
pl.ylabel('max absolute error (K)')
pl.title('Grid refinement study')
pl.grid()
pl.show()

In [ ]:
# In this section I am celebrating
print('CW done: I deserve a good mark')
print("This is shiv, wasssuppp")